# Task 3 — Ball Detection & Table Retrieval

This notebook covers the full pipeline for **ball detection** on pool/billiard table images
using six object-detection architectures, followed by an image-retrieval system based on
ball-position similarity.

---

## Table of Contents

1. [Setup & Imports](#1-setup--imports)
2. [Data Preparation](#2-data-preparation)
3. [Ball Detection](#3-ball-detection)
   - [3.1 YOLOv8 — Without Table Masks](#31-yolov8--without-table-masks)
   - [3.2 YOLOv8 — With Table Masks](#32-yolov8--with-table-masks)
   - [3.3 RT-DETR](#33-rt-detr)
   - [3.4 DETR (HuggingFace)](#34-detr-huggingface)
   - [3.5 Faster R-CNN](#35-faster-r-cnn)
   - [3.6 SSD](#36-ssd)
   - [3.7 Quantitative Comparison](#37-quantitative-comparison)
4. [Table Retrieval](#4-table-retrieval)
   - [4.1 Ball Position Extraction](#41-ball-position-extraction)
   - [4.2 Normalized Court Mapping via Homography](#42-normalized-court-mapping-via-homography)
   - [4.3 Similarity Search (EMD)](#43-similarity-search-emd)
   - [4.4 Qualitative Evaluation](#44-qualitative-evaluation)

---

## 1 — Setup & Imports

- **ultralytics** — YOLOv8, RT-DETR  
- **transformers** — DETR (`facebook/detr-resnet-50`)  
- **torchvision** — Faster R-CNN, SSD  
- **scipy** — Earth Mover's Distance  
- **opencv-python** — masking & homography

In [1]:

import json, os, random, shutil, glob, warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import cv2
import torch
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights,
    ssd300_vgg16, SSD300_VGG16_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssd import SSDClassificationHead
from PIL import Image
from tqdm import tqdm
from scipy.stats import wasserstein_distance
from transformers import DetrImageProcessor, DetrForObjectDetection
from ultralytics import YOLO, RTDETR

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device     : {DEVICE}")
print(f"PyTorch    : {torch.__version__}")
print(f"torchvision: {torchvision.__version__}")

c:\Users\João Sousa\Documents\GitHub\Computer-Vision\cv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device     : cuda
PyTorch    : 2.12.0+cu132
torchvision: 0.27.0+cu132


In [2]:
# ── Checkpoint helpers ───────────────────────────────────────────────────────
CHECKPOINT_FILE = "task3_checkpoints.json"


def save_checkpoint(key: str, results: dict):
    """Persiste os resultados de um modelo no ficheiro de checkpoints."""
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE) as _f:
            ckpt = json.load(_f)
    else:
        ckpt = {}
    ckpt[key] = results
    with open(CHECKPOINT_FILE, "w") as _f:
        json.dump(ckpt, _f, indent=2)
    print(f"  ✓ checkpoint saved → {CHECKPOINT_FILE} [{key}]")


def load_checkpoints() -> dict:
    """Lê todos os checkpoints guardados."""
    if not os.path.exists(CHECKPOINT_FILE):
        return {}
    with open(CHECKPOINT_FILE) as _f:
        return json.load(_f)


print("Checkpoint helpers ready.")

Checkpoint helpers ready.


---
## 2 — Data Preparation

Converts `unified_dataset.json` into two YOLO directory trees:

```
yolo_data/
  pretrain/   ← datasets 1 + 3  (90 % train / 10 % val)
  finetune/   ← dataset 4       (official train / val / test)
```

Dataset 2 is **skipped** (invalid images). All boxes → `class 0 = ball`.

Helper functions (`coco_to_yolo`, `write_label`, `setup_dirs`, `write_yaml`) defined here
are reused later for the **masked** dataset in §3.2.

In [3]:
# ── Shared helpers (also used by masked-data preparation in §3.2) ─────────────

UNIFIED_JSON = "unified_dataset.json"
SKIP_DS_ID   = 2
ORIG_DS_ID   = 4
PRETRAIN_VAL = 0.1
SEED         = 42


def coco_to_yolo(bbox, img_w, img_h):
    """COCO [x,y,w,h] → YOLO [xc,yc,w,h] normalised."""
    x, y, w, h = [float(v) for v in bbox]
    return (
        max(0.0, min(1.0, (x + w / 2) / img_w)),
        max(0.0, min(1.0, (y + h / 2) / img_h)),
        max(0.0, min(1.0, w / img_w)),
        max(0.0, min(1.0, h / img_h)),
    )


def write_label(label_path, bboxes, img_w, img_h):
    """Write a YOLO label file — all boxes are class 0."""
    lines = []
    for bbox in bboxes:
        xc, yc, wn, hn = coco_to_yolo(bbox, img_w, img_h)
        lines.append(f"0 {xc:.6f} {yc:.6f} {wn:.6f} {hn:.6f}")
    with open(label_path, "w") as f:
        f.write("\n".join(lines))


def setup_dirs(base):
    """Create images/{train,val,test} and labels/{train,val,test} under base."""
    for split in ["train", "val", "test"]:
        os.makedirs(os.path.join(base, "images", split), exist_ok=True)
        os.makedirs(os.path.join(base, "labels", split), exist_ok=True)


def write_yaml(path, data_dir, splits, nc=1, names=None):
    """Write a YOLO dataset YAML file."""
    if names is None:
        names = ["ball"]
    with open(path, "w", encoding="utf-8") as f:
        f.write(f"path: {os.path.abspath(data_dir)}\n")
        for s in ["train", "val", "test"]:
            if s in splits:
                f.write(f"{s}: images/{s}\n")
            else:
                f.write(f"# {s}: not available\n")
        f.write(f"\nnc: {nc}\nnames: {names}\n")


def process_entry(entry, images_dir, labels_dir):
    """Copy original image + write YOLO label. Returns True on success."""
    src = entry["image_path"]
    if not os.path.isfile(src):
        return False
    fname = entry["file_name"]
    shutil.copy2(src, os.path.join(images_dir, fname))
    write_label(
        os.path.join(labels_dir, os.path.splitext(fname)[0] + ".txt"),
        [a["bbox"] for a in entry["annotations"]],
        int(entry["width"]), int(entry["height"]),
    )
    return True


print("Shared helpers defined.")

Shared helpers defined.


In [4]:
# ── Build yolo_data/ (unmasked) ───────────────────────────────────────────────

YOLO_DATA_DIR = "yolo_data"
PRETRAIN_YAML = os.path.join(YOLO_DATA_DIR, "pretrain.yaml")
FINETUNE_YAML = os.path.join(YOLO_DATA_DIR, "finetune.yaml")

with open(UNIFIED_JSON) as f:
    ALL_ENTRIES = json.load(f)

pretrain_dir = os.path.join(YOLO_DATA_DIR, "pretrain")
finetune_dir = os.path.join(YOLO_DATA_DIR, "finetune")
setup_dirs(pretrain_dir)
setup_dirs(finetune_dir)

aux_entries = [e for e in ALL_ENTRIES if e["dataset"] not in (SKIP_DS_ID, ORIG_DS_ID)]
random.seed(SEED)
random.shuffle(aux_entries)
val_n   = max(1, int(PRETRAIN_VAL * len(aux_entries)))
aux_val = set(id(e) for e in aux_entries[:val_n])

counts = {"pretrain": {"train": 0, "val": 0}, "finetune": {"train": 0, "val": 0, "test": 0}}
ds_counts = defaultdict(int)

for entry in ALL_ENTRIES:
    ds, spl = entry["dataset"], entry.get("split", "train")
    if ds == SKIP_DS_ID:
        continue
    if ds != ORIG_DS_ID:
        split = "val" if id(entry) in aux_val else "train"
        if process_entry(entry,
                         os.path.join(pretrain_dir, "images", split),
                         os.path.join(pretrain_dir, "labels", split)):
            counts["pretrain"][split] += 1
            ds_counts[f"ds{ds}"] += 1
    else:
        if spl not in ("train", "val", "test"):
            continue
        if process_entry(entry,
                         os.path.join(finetune_dir, "images", spl),
                         os.path.join(finetune_dir, "labels", spl)):
            counts["finetune"][spl] += 1
            ds_counts[f"ds{ds}"] += 1

write_yaml(PRETRAIN_YAML, pretrain_dir, splits=["train", "val"])
write_yaml(FINETUNE_YAML, finetune_dir, splits=["train", "val", "test"])

print("yolo_data/ ready.")
print(f"  Pretrain (ds1+3) train: {counts['pretrain']['train']}  val: {counts['pretrain']['val']}")
for ds_key, n in sorted(ds_counts.items()):
    print(f"    {ds_key}: {n} images")
print(f"  Finetune (ds4)   train: {counts['finetune']['train']}  "
      f"val: {counts['finetune']['val']}  test: {counts['finetune']['test']}")

yolo_data/ ready.
  Pretrain (ds1+3) train: 828  val: 91
    ds1: 485 images
    ds3: 434 images
    ds4: 247 images
  Finetune (ds4)   train: 194  val: 26  test: 27


---
## 3 — Ball Detection

| # | Model | Framework | Training strategy |
|---|-------|-----------|-------------------|
| 3.1 | YOLOv8s (no mask) | ultralytics | Pretrain ds1+3 → finetune ds4 |
| 3.2 | YOLOv8s (mask) | ultralytics | Same pipeline on masked images |
| 3.3 | RT-DETR-L | ultralytics | Same 2-phase pipeline |
| 3.4 | DETR ResNet-50 | HuggingFace | Custom loop, ds4 only |
| 3.5 | Faster R-CNN | torchvision | Custom loop, ds4 only |
| 3.6 | SSD300 | torchvision | Custom loop, ds4 only |

### 3.1 YOLOv8 — Without Table Masks

**Phase 1** — Pretrain on datasets 1 + 3 (`pretrain.yaml`).  
**Phase 2** — Fine-tune on dataset 4 (`finetune.yaml`).

In [7]:


MODELS_DIR      = "models"
RESULTS_FILE    = "task3_results.txt"
BASE_WEIGHTS    = "yolov8s.pt"
IMG_SIZE        = 640
BATCH_SIZE      = 32
PATIENCE        = 15
PRETRAIN_EPOCHS = 30
FINETUNE_EPOCHS = 100

os.makedirs(MODELS_DIR, exist_ok=True)


def find_best(run_name):
    """Locate best.pt for a run regardless of ultralytics nesting depth."""
    matches = glob.glob(f"**/{run_name}/weights/best.pt", recursive=True)
    if not matches:
        raise FileNotFoundError(f"best.pt not found for run: {run_name}")
    return max(matches, key=os.path.getmtime)


print("YOLO config ready.")

YOLO config ready.


In [6]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [7]:
# ── Phase 1: Pretrain on datasets 1 + 3 (no mask) ────────────────────────────
print("=" * 55)
print("Phase 1 — Pretrain on aux datasets (1, 3) — NO MASK")
print("=" * 55)

pretrain_model = YOLO(BASE_WEIGHTS)
pretrain_model.train(
    data=PRETRAIN_YAML, epochs=PRETRAIN_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    workers=4, cache=False, device=0,
    name="yolov8s_pretrain", exist_ok=True, single_cls=True,
)
pretrain_best = find_best("yolov8s_pretrain")
shutil.copy(pretrain_best, os.path.join(MODELS_DIR, "yolov8s_pretrained.pt"))
print(f"Pretrained weights -> {MODELS_DIR}/yolov8s_pretrained.pt  (src: {pretrain_best})")

Phase 1 — Pretrain on aux datasets (1, 3) — NO MASK
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\pretrain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_pretrain, nbs=64, nms=False, opset=

In [8]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [9]:
# ── Phase 2: Fine-tune on dataset 4 (no mask) ────────────────────────────────
print("=" * 55)
print("Phase 2 — Fine-tune on dataset 4 — NO MASK")
print("=" * 55)

finetune_model = YOLO(pretrain_best)
finetune_model.train(
    data=FINETUNE_YAML, epochs=FINETUNE_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    workers=4, cache=False, device=0,
    name="yolov8s_finetune", exist_ok=True, single_cls=True,
    lr0=1e-4, lrf=0.01,
)
finetune_best = find_best("yolov8s_finetune")
shutil.copy(finetune_best, os.path.join(MODELS_DIR, "yolov8s_final.pt"))
print(f"Fine-tuned weights -> {MODELS_DIR}/yolov8s_final.pt  (src: {finetune_best})")

Phase 2 — Fine-tune on dataset 4 — NO MASK
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data\finetune.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\yolov8s_pretrain\weights\best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_finetun

In [10]:
# ── Evaluation: dataset 4 test split (no mask) ───────────────────────────────
eval_model      = YOLO(finetune_best)
metrics_no_mask = eval_model.val(data=FINETUNE_YAML, split="test",
                                  imgsz=IMG_SIZE, batch=BATCH_SIZE)
yolo_no_mask_results = {
    "mAP50":     float(metrics_no_mask.box.map50),
    "mAP50-95":  float(metrics_no_mask.box.map),
    "Precision": float(metrics_no_mask.box.mp),
    "Recall":    float(metrics_no_mask.box.mr),
}
print("YOLOv8s (no mask):", yolo_no_mask_results)

save_checkpoint("yolo_no_mask_results", yolo_no_mask_results)

Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 238.617.2 MB/s, size: 1390.7 KB)
val: Scanning C:\Users\João Sousa\Documents\GitHub\Computer-Vision\yolo_data\finetune\labels\test.cache... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 14.2Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 2.3s/it 2.3s
                   all         27        309       0.99      0.947      0.974        0.8
Speed: 1.9ms preprocess, 6.5ms inference, 0.0ms loss, 0.6ms postprocess per image
Results saved to C:\Users\Joo Sousa\Documents\GitHub\Computer-Vision\runs\detect\val-5
YOLOv8s (no mask): {'mAP50': 0.9738119945664387, 'mAP50-95': 0.800218902055421, 'Precision': 0.9898539758824394, 'Recall': 0.9471943625184456}
  ✓ checkpoint saved → task3

In [11]:
# ── Sample predictions (no mask) ─────────────────────────────────────────────
test_images_dir = os.path.join(YOLO_DATA_DIR, "finetune", "images", "test")
test_imgs = sorted(
    list(Path(test_images_dir).glob("*.jpg")) +
    list(Path(test_images_dir).glob("*.png"))
)[:6]

if test_imgs:
    preds = eval_model.predict(source=[str(p) for p in test_imgs],
                               imgsz=IMG_SIZE, conf=0.25, verbose=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, r in zip(axes.flatten(), preds):
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"{Path(r.path).name}\n{len(r.boxes)} balls", fontsize=8)
        ax.axis("off")
    for ax in axes.flatten()[len(preds):]:
        ax.axis("off")
    plt.suptitle("YOLOv8s — No Mask — Test Predictions")
    plt.tight_layout()
    plt.savefig("task3_yolo_no_mask_predictions.png", dpi=150)
    print("Saved -> task3_yolo_no_mask_predictions.png")

Saved -> task3_yolo_no_mask_predictions.png


---
### 3.2 YOLOv8 — With Table Masks

The **same two-phase pipeline** is repeated on masked images.  
The `segment_table` function detects the billiard table cloth via HSV colour ranges
(blue/green) and returns a binary mask + 4 corner coordinates.  
Background pixels are zeroed out before saving.

Masking is applied to **all three splits across both datasets**:

| Source | Destination | Description |
|--------|-------------|-------------|
| Datasets 1 & 3 (pretrain) | `yolo_data_masked/pretrain/` | Masked pretrain images |
| Dataset 4 (finetune) | `yolo_data_masked/finetune/` | Masked fine-tune images |

In [12]:
# ── Table segmentation helpers ────────────────────────────────────────────────

def order_corners(corners):
    """Order 4 corners as top-left, top-right, bottom-right, bottom-left."""
    corners = np.array(corners, dtype=np.float32)
    center  = corners.mean(axis=0)
    angles  = np.arctan2(corners[:, 1] - center[1], corners[:, 0] - center[0])
    order   = np.argsort(angles)
    corners = corners[order]
    top_idx = np.argmin(corners[:, 0] + corners[:, 1])
    corners = np.roll(corners, -top_idx, axis=0)
    return corners


def segment_table(img):
    """Detect the billiard table cloth and return (mask, corners).
    Iterates contours by area and picks the first one with uniform color
    (low H std inside the contour), rejecting banners and backgrounds.
    """
    h_img, w_img = img.shape[:2]
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    blue_mask  = cv2.inRange(hsv, np.array([90, 80, 80]),  np.array([130, 255, 255]))
    green_mask = cv2.inRange(hsv, np.array([35, 80,  50]), np.array([ 95, 255, 255]))
    mask = cv2.bitwise_or(blue_mask, green_mask)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        full    = np.ones(img.shape[:2], dtype=np.uint8) * 255
        corners = np.array([[0, 0], [w_img, 0], [w_img, h_img], [0, h_img]], dtype=np.float32)
        return full, corners

    contours  = sorted(contours, key=cv2.contourArea, reverse=True)
    h_channel = hsv[:, :, 0]

    for contour in contours[:5]:
        epsilon = 0.02 * cv2.arcLength(contour, True)
        approx  = cv2.approxPolyDP(contour, epsilon, True)
        if len(approx) == 4:
            corners = approx.reshape(4, 2).astype(np.float32)
        else:
            rect    = cv2.minAreaRect(contour)
            corners = cv2.boxPoints(rect).astype(np.float32)

        candidate_mask = np.zeros(img.shape[:2], dtype=np.uint8)
        cv2.fillPoly(candidate_mask, [corners.astype(np.int32)], 255)
        pixels = h_channel[candidate_mask == 255]
        if len(pixels) == 0:
            continue
        if pixels.std() < 40:
            corners    = order_corners(corners)
            clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
            cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
            return clean_mask, corners

    # Fallback — use largest contour
    contour = contours[0]
    epsilon = 0.02 * cv2.arcLength(contour, True)
    approx  = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) == 4:
        corners = approx.reshape(4, 2).astype(np.float32)
    else:
        rect    = cv2.minAreaRect(contour)
        corners = cv2.boxPoints(rect).astype(np.float32)
    corners    = order_corners(corners)
    clean_mask = np.zeros(img.shape[:2], dtype=np.uint8)
    cv2.fillPoly(clean_mask, [corners.astype(np.int32)], 255)
    return clean_mask, corners




def apply_table_mask(img):
    """Zero-out background outside the table. Falls back to original if mask is too small."""
    mask, _ = segment_table(img)
    masked  = cv2.bitwise_and(img, img, mask=mask)
    # Fallback: mask covers < 10 % of image → likely failed
    if np.count_nonzero(masked) < 0.10 * img.size:
        return img
    return masked


print("segment_table / apply_table_mask ready.")

segment_table / apply_table_mask ready.


In [13]:
# ============================================================
# PREPARE MASKED DATA
#
# segment_table() is applied to EVERY image across ALL datasets:
#
#   Datasets 1 & 3  →  yolo_data_masked/pretrain/  (Phase 1 training)
#   Dataset 4       →  yolo_data_masked/finetune/   (Phase 2 training + eval)
#
# Labels are copied unchanged — bounding boxes are unaffected by masking.
# ============================================================

YOLO_MASKED_DIR      = "yolo_data_masked"
PRETRAIN_MASKED_YAML = os.path.join(YOLO_MASKED_DIR, "pretrain.yaml")
FINETUNE_MASKED_YAML = os.path.join(YOLO_MASKED_DIR, "finetune.yaml")


def process_entry_masked(entry, images_dir, labels_dir):
    """
    Apply table mask to image, save to images_dir.
    Copy YOLO label unchanged to labels_dir.
    """
    src = entry["image_path"]
    if not os.path.isfile(src):
        return False
    fname     = entry["file_name"]
    dst_img   = os.path.join(images_dir, fname)
    dst_label = os.path.join(labels_dir, os.path.splitext(fname)[0] + ".txt")

    img = cv2.imread(src)
    if img is not None:
        cv2.imwrite(dst_img, apply_table_mask(img))   # ← segment_table applied here
    else:
        shutil.copy2(src, dst_img)                    # fallback: copy original

    write_label(dst_label,
                [a["bbox"] for a in entry["annotations"]],
                int(entry["width"]), int(entry["height"]))
    return True


def prepare_masked_data():
    with open(UNIFIED_JSON) as f:
        entries = json.load(f)

    # Destination directories
    masked_pretrain_dir = os.path.join(YOLO_MASKED_DIR, "pretrain")
    masked_finetune_dir = os.path.join(YOLO_MASKED_DIR, "finetune")
    setup_dirs(masked_pretrain_dir)
    setup_dirs(masked_finetune_dir)

    # Build 90/10 split for auxiliary datasets (1, 3)
    aux_entries = [e for e in entries
                   if e["dataset"] not in (SKIP_DS_ID, ORIG_DS_ID)]
    random.seed(SEED)
    random.shuffle(aux_entries)
    val_n   = max(1, int(PRETRAIN_VAL * len(aux_entries)))
    aux_val = set(id(e) for e in aux_entries[:val_n])

    counts    = {"pretrain": {"train": 0, "val": 0},
                 "finetune": {"train": 0, "val": 0, "test": 0}}
    ds_counts = defaultdict(int)

    for entry in entries:
        ds  = entry["dataset"]
        spl = entry.get("split", "train")

        if ds == SKIP_DS_ID:          # skip dataset 2
            continue

        if ds != ORIG_DS_ID:
            # ── Datasets 1 & 3 → masked PRETRAIN ──────────────────────────
            aux_split = "val" if id(entry) in aux_val else "train"
            ok = process_entry_masked(
                entry,
                os.path.join(masked_pretrain_dir, "images", aux_split),
                os.path.join(masked_pretrain_dir, "labels", aux_split),
            )
            if ok:
                counts["pretrain"][aux_split] += 1
                ds_counts[f"ds{ds} → pretrain/{aux_split}"] += 1
        else:
            # ── Dataset 4 → masked FINETUNE ───────────────────────────────
            if spl not in ("train", "val", "test"):
                continue
            ok = process_entry_masked(
                entry,
                os.path.join(masked_finetune_dir, "images", spl),
                os.path.join(masked_finetune_dir, "labels", spl),
            )
            if ok:
                counts["finetune"][spl] += 1
                ds_counts[f"ds{ds} → finetune/{spl}"] += 1

    write_yaml(PRETRAIN_MASKED_YAML, masked_pretrain_dir, splits=["train", "val"])
    write_yaml(FINETUNE_MASKED_YAML, masked_finetune_dir, splits=["train", "val", "test"])

    print(f"yolo_data_masked/ ready.")
    print(f"  [PRETRAIN — ds1+3 masked]")
    print(f"    train : {counts['pretrain']['train']}")
    print(f"    val   : {counts['pretrain']['val']}")
    print(f"  [FINETUNE — ds4 masked]")
    print(f"    train : {counts['finetune']['train']}")
    print(f"    val   : {counts['finetune']['val']}")
    print(f"    test  : {counts['finetune']['test']}")
    print("  Per-split breakdown:")
    for k, v in sorted(ds_counts.items()):
        print(f"    {k}: {v}")


prepare_masked_data()

yolo_data_masked/ ready.
  [PRETRAIN — ds1+3 masked]
    train : 828
    val   : 91
  [FINETUNE — ds4 masked]
    train : 194
    val   : 26
    test  : 27
  Per-split breakdown:
    ds1 → pretrain/train: 443
    ds1 → pretrain/val: 42
    ds3 → pretrain/train: 385
    ds3 → pretrain/val: 49
    ds4 → finetune/test: 27
    ds4 → finetune/train: 194
    ds4 → finetune/val: 26


In [14]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [15]:
# ── Phase 1: Pretrain on masked datasets 1 + 3 ───────────────────────────────
print("=" * 55)
print("Phase 1 — Pretrain on aux datasets (1, 3) — WITH MASK")
print("=" * 55)

pretrain_model_m = YOLO(BASE_WEIGHTS)
pretrain_model_m.train(
    data=PRETRAIN_MASKED_YAML, epochs=PRETRAIN_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    workers=4, cache=False, device=0,
    name="yolov8s_pretrain_masked", exist_ok=True, single_cls=True,
)
pretrain_best_m = find_best("yolov8s_pretrain_masked")
shutil.copy(pretrain_best_m, os.path.join(MODELS_DIR, "yolov8s_pretrained_masked.pt"))
print(f"Masked pretrain weights -> {MODELS_DIR}/yolov8s_pretrained_masked.pt")
print(f"  (src: {pretrain_best_m})")

Phase 1 — Pretrain on aux datasets (1, 3) — WITH MASK
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data_masked\pretrain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_pretrain_masked, nbs=64, n

In [16]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [17]:
# ── Phase 2: Fine-tune on masked dataset 4 ───────────────────────────────────
print("=" * 55)
print("Phase 2 — Fine-tune on dataset 4 — WITH MASK")
print("=" * 55)

finetune_model_m = YOLO(pretrain_best_m)
finetune_model_m.train(
    data=FINETUNE_MASKED_YAML, epochs=FINETUNE_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    workers=4, cache=False, device=0,
    name="yolov8s_finetune_masked", exist_ok=True, single_cls=True,
    lr0=1e-4, lrf=0.01,
)
finetune_best_m = find_best("yolov8s_finetune_masked")
shutil.copy(finetune_best_m, os.path.join(MODELS_DIR, "yolov8s_final_masked.pt"))
print(f"Masked fine-tuned weights -> {MODELS_DIR}/yolov8s_final_masked.pt")
print(f"  (src: {finetune_best_m})")

Phase 2 — Fine-tune on dataset 4 — WITH MASK
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data_masked\finetune.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=runs\detect\yolov8s_pretrain_masked\weights\best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name

In [18]:
# ── Evaluation: dataset 4 test split (with mask) ─────────────────────────────
eval_model_m = YOLO(finetune_best_m)
metrics_mask = eval_model_m.val(data=FINETUNE_MASKED_YAML, split="test",
                                 imgsz=IMG_SIZE, batch=BATCH_SIZE)
yolo_mask_results = {
    "mAP50":     float(metrics_mask.box.map50),
    "mAP50-95":  float(metrics_mask.box.map),
    "Precision": float(metrics_mask.box.mp),
    "Recall":    float(metrics_mask.box.mr),
}
print("YOLOv8s (masked):", yolo_mask_results)

save_checkpoint("yolo_mask_results", yolo_mask_results)

Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access  (ping: 0.00.0 ms, read: 50.218.7 MB/s, size: 225.7 KB)
val: Scanning C:\Users\João Sousa\Documents\GitHub\Computer-Vision\yolo_data_masked\finetune\labels\test... 27 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 27/27 654.4it/s 0.0s
val: New cache created: C:\Users\Joo Sousa\Documents\GitHub\Computer-Vision\yolo_data_masked\finetune\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 1/1 1.9s/it 1.9s
                   all         27        309      0.997      0.951      0.965      0.804
Speed: 0.9ms preprocess, 4.9ms inference, 0.0ms loss, 1.1ms postprocess per image
Results saved to C:\Users\Joo Sousa\Documents\GitHub\Computer-Vision\runs\detect\val-6
YOLOv8s (masked): {'mAP50': 0.9645806451612903, 'mAP

In [19]:
# ── Sample predictions (with mask) ───────────────────────────────────────────
test_imgs_m = sorted(
    list(Path(os.path.join(YOLO_MASKED_DIR, "finetune", "images", "test")).glob("*.jpg")) +
    list(Path(os.path.join(YOLO_MASKED_DIR, "finetune", "images", "test")).glob("*.png"))
)[:6]

if test_imgs_m:
    preds_m = eval_model_m.predict(source=[str(p) for p in test_imgs_m],
                                    imgsz=IMG_SIZE, conf=0.25, verbose=False)
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, r in zip(axes.flatten(), preds_m):
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"{Path(r.path).name}\n{len(r.boxes)} balls", fontsize=8)
        ax.axis("off")
    for ax in axes.flatten()[len(preds_m):]:
        ax.axis("off")
    plt.suptitle("YOLOv8s — With Mask — Test Predictions")
    plt.tight_layout()
    plt.savefig("task3_yolo_masked_predictions.png", dpi=150)
    print("Saved -> task3_yolo_masked_predictions.png")

Saved -> task3_yolo_masked_predictions.png


In [20]:
# ── YOLOv8 Mask vs No-Mask comparison ────────────────────────────────────────
yolo_both = {
    "YOLOv8s (no mask)": yolo_no_mask_results,
    "YOLOv8s (masked)":  yolo_mask_results,
}
print(f"\n{'='*60}")
print(f"{'Model':<22} | {'mAP@50':>8} | {'mAP@50-95':>10} | {'Precision':>10} | {'Recall':>8}")
print(f"{'-'*60}")
for name, r in yolo_both.items():
    print(f"{name:<22} | {r['mAP50']:>8.4f} | {r['mAP50-95']:>10.4f} | "
          f"{r['Precision']:>10.4f} | {r['Recall']:>8.4f}")
print("=" * 60)

labels  = ["mAP@50", "mAP@50-95", "Precision", "Recall"]
keys    = ["mAP50", "mAP50-95", "Precision", "Recall"]
x, w    = np.arange(len(labels)), 0.35
fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - w/2, [yolo_no_mask_results[k] for k in keys], w,
       label="No mask", color="steelblue", alpha=0.85)
ax.bar(x + w/2, [yolo_mask_results[k]    for k in keys], w,
       label="Masked",  color="coral",     alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylim(0, 1.1); ax.set_ylabel("Score")
ax.set_title("YOLOv8s — Mask vs No-Mask (test split)")
ax.legend(); plt.tight_layout()
plt.savefig("task3_yolo_comparison.png", dpi=150)
print("Saved -> task3_yolo_comparison.png")


Model                  |   mAP@50 |  mAP@50-95 |  Precision |   Recall
------------------------------------------------------------
YOLOv8s (no mask)      |   0.9738 |     0.8002 |     0.9899 |   0.9472
YOLOv8s (masked)       |   0.9646 |     0.8038 |     0.9970 |   0.9515
Saved -> task3_yolo_comparison.png


---
### 3.3 RT-DETR

Real-Time DEtection TRansformer via `ultralytics` (`YOLO("rtdetr-l.pt")`).  
Same two-phase pipeline as YOLOv8 (pretrain ds1+3 → finetune ds4).

In [23]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [34]:

import os, shutil
from pathlib import Path
from ultralytics import RTDETR

# -- Paths
MODELS_DIR    = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

PRETRAIN_YAML = "yolo_data/pretrain.yaml"
FINETUNE_YAML = "yolo_data/finetune.yaml"

# -- Hyperparameters
PRETRAIN_EPOCHS = 50
FINETUNE_EPOCHS = 100
IMG_SIZE        = 640
BATCH_SIZE      = 4   # increase if GPU has more VRAM (e.g. 32 for 24GB)
PATIENCE        = 20
SEED            = 42

def find_best(run_name):
    p = Path(f"runs/detect/{run_name}/weights/best.pt")
    if p.exists():
        return str(p)
    raise FileNotFoundError(f"best.pt not found for run: {run_name}")

# -- Phase 1: pretrain on datasets 1+3
print("=" * 55)
print("RT-DETR — Phase 1: pretrain on datasets 1, 3")
print("=" * 55)
rtdetr_pretrain = RTDETR("rtdetr-l.pt")
rtdetr_pretrain.train(
    data=PRETRAIN_YAML, epochs=PRETRAIN_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    name="rtdetr_pretrain", exist_ok=True, single_cls=True,
    device=0,        # GPU 0
    workers=4,
    cache=False,
)
rtdetr_pretrain_best = find_best("rtdetr_pretrain")
shutil.copy(rtdetr_pretrain_best, os.path.join(MODELS_DIR, "rtdetr_pretrained.pt"))

# -- Phase 2: fine-tune on dataset 4
print("=" * 55)
print("RT-DETR — Phase 2: fine-tune on dataset 4")
print("=" * 55)
rtdetr_finetune = RTDETR(rtdetr_pretrain_best)
rtdetr_finetune.train(
    data=FINETUNE_YAML, epochs=FINETUNE_EPOCHS, imgsz=IMG_SIZE,
    batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
    name="rtdetr_finetune", exist_ok=True, single_cls=True,
    device=0,
    workers=4,
    cache=False,
    lr0=1e-4, lrf=0.01,
)
rtdetr_best = find_best("rtdetr_finetune")
shutil.copy(rtdetr_best, os.path.join(MODELS_DIR, "rtdetr_final.pt"))

# -- Evaluation
eval_rtdetr = RTDETR(rtdetr_best)
rt_metrics  = eval_rtdetr.val(
    data=FINETUNE_YAML, split="test",
    imgsz=IMG_SIZE, batch=BATCH_SIZE,
    device=0,
)
rtdetr_results = {
    "mAP50":     float(rt_metrics.box.map50),
    "mAP50-95":  float(rt_metrics.box.map),
    "Precision": float(rt_metrics.box.mp),
    "Recall":    float(rt_metrics.box.mr),
}
print("RT-DETR results:", rtdetr_results)

RT-DETR — Phase 1: pretrain on datasets 1, 3
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data/pretrain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=rtdetr_pretrain, nbs=64, nms=False, opset=None, op

---
### 3.4 DETR (HuggingFace)

`facebook/detr-resnet-50` fine-tuned on dataset 4 with a custom training loop.  
Boxes must be centre-format `[cx, cy, w, h]` normalised to `[0, 1]`.

In [27]:
# ── Shared PyTorch dataset & evaluation utilities ─────────────────────────────

IMG_TRANSFORM = transforms.Compose([transforms.ToTensor()])

with open(UNIFIED_JSON) as f:
    _all = json.load(f)
for _e in _all:
    for _a in _e.get("annotations", []):
        _a["bbox"] = [float(v) for v in _a["bbox"]]
ds4_train = [e for e in _all if e["dataset"] == 4 and e["split"] == "train"]
ds4_val   = [e for e in _all if e["dataset"] == 4 and e["split"] == "val"]
ds4_test  = [e for e in _all if e["dataset"] == 4 and e["split"] == "test"]
print(f"DS4 → train:{len(ds4_train)}  val:{len(ds4_val)}  test:{len(ds4_test)}")


class BallDataset(Dataset):
    def __init__(self, entries, tfm=None):
        self.entries = [e for e in entries if os.path.isfile(e["image_path"])]
        self.tfm     = tfm

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        e = self.entries[idx]
        img = Image.open(e["image_path"]).convert("RGB")
        W, H = img.size
        boxes, labels = [], []
        for a in e["annotations"]:
            x, y, w, h = (float(v) for v in a["bbox"])
            x1, y1 = max(0., x),          max(0., y)
            x2, y2 = min(float(W), x + w), min(float(H), y + h)
            if x2 > x1 and y2 > y1:
                boxes.append([x1, y1, x2, y2]); labels.append(1)
        boxes  = torch.as_tensor(boxes,  dtype=torch.float32) if boxes  else torch.zeros((0, 4))
        labels = torch.as_tensor(labels, dtype=torch.int64)   if labels else torch.zeros((0,), dtype=torch.int64)
        if self.tfm: img = self.tfm(img)
        return img, {"boxes": boxes, "labels": labels, "image_id": torch.tensor([idx])}


def collate_fn(batch): return tuple(zip(*batch))


train_loader = DataLoader(BallDataset(ds4_train, IMG_TRANSFORM), batch_size=4,
                          shuffle=True,  collate_fn=collate_fn)
val_loader   = DataLoader(BallDataset(ds4_val,   IMG_TRANSFORM), batch_size=4,
                          shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(BallDataset(ds4_test,  IMG_TRANSFORM), batch_size=1,
                          shuffle=False, collate_fn=collate_fn)


# ── Evaluation helpers ────────────────────────────────────────────────────────

def iou(a, b):
    xa,ya = max(a[0],b[0]), max(a[1],b[1])
    xb,yb = min(a[2],b[2]), min(a[3],b[3])
    inter = max(0,xb-xa)*max(0,yb-ya)
    union = (a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return inter/union if union>0 else 0.


def evaluate_detections(all_preds, all_gts, iou_thresholds=None):
    if iou_thresholds is None:
        iou_thresholds = np.arange(0.50, 1.00, 0.05).tolist()
    APs_50, APs_all = [], []
    all_tp, all_fp, all_fn = 0, 0, 0
    for thr in iou_thresholds:
        tp_l, fp_l, sc_l, n_gt = [], [], [], 0
        for preds, gts in zip(all_preds, all_gts):
            pb, ps, gb = preds["boxes"], preds["scores"], gts["boxes"]
            n_gt += len(gb); matched = set()
            for i in (np.argsort(-ps) if len(ps) > 0 else []):
                bst_iou, bst_j = 0., -1
                for j, g in enumerate(gb):
                    if j in matched: continue
                    v = iou(pb[i], g)
                    if v > bst_iou: bst_iou, bst_j = v, j
                if bst_iou >= thr:
                    tp_l.append(1); fp_l.append(0); matched.add(bst_j)
                else:
                    tp_l.append(0); fp_l.append(1)
                sc_l.append(ps[i])
        if not sc_l: APs_all.append(0.); continue
        order  = np.argsort(-np.array(sc_l))
        tp_cum = np.cumsum(np.array(tp_l)[order])
        fp_cum = np.cumsum(np.array(fp_l)[order])
        rec    = tp_cum / max(n_gt, 1)
        prec   = tp_cum / np.maximum(tp_cum + fp_cum, 1)
        ap     = sum(prec[rec >= t].max() if (rec >= t).any() else 0.
                     for t in np.linspace(0,1,11)) / 11
        if abs(thr - 0.50) < 1e-6:
            APs_50.append(ap)
            all_tp += int(tp_cum[-1]); all_fp += int(fp_cum[-1])
            all_fn += n_gt - int(tp_cum[-1])
        APs_all.append(ap)
    return {
        "mAP50":     float(np.mean(APs_50)),
        "mAP50-95":  float(np.mean(APs_all)),
        "Precision": float(all_tp / max(all_tp + all_fp, 1)),
        "Recall":    float(all_tp / max(all_tp + all_fn, 1)),
    }


def run_inference(model, loader, conf=0.25):
    model.eval(); all_preds, all_gts = [], []
    with torch.no_grad():
        for imgs, targets in tqdm(loader, desc="Inference"):
            for out, tgt in zip(model([img.to(DEVICE) for img in imgs]), targets):
                mask = out["scores"] >= conf
                all_preds.append({"boxes":  out["boxes"][mask].cpu().numpy(),
                                  "scores": out["scores"][mask].cpu().numpy(),
                                  "labels": out["labels"][mask].cpu().numpy()})
                all_gts.append({"boxes": tgt["boxes"].numpy()})
    return all_preds, all_gts


print("BallDataset & evaluation helpers ready.")

DS4 → train:194  val:26  test:27
BallDataset & evaluation helpers ready.


In [28]:
from transformers import DetrImageProcessor, DetrForObjectDetection

DETR_PROC = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")


class DetrBallDataset(Dataset):
    def __init__(self, entries, proc):
        self.entries = [e for e in entries if os.path.isfile(e["image_path"])]
        self.proc    = proc

    def __len__(self): return len(self.entries)

    def __getitem__(self, idx):
        e   = self.entries[idx]
        img = Image.open(e["image_path"]).convert("RGB")
        anns = []
        for a in e["annotations"]:
            bb = [float(v) for v in a["bbox"]]
            if bb[2] > 0 and bb[3] > 0:
                anns.append({"bbox": bb, "category_id": 0,
                             "area": bb[2] * bb[3], "iscrowd": 0})
        enc = self.proc(images=img,
                        annotations={"image_id": idx, "annotations": anns},
                        return_tensors="pt")
        return enc["pixel_values"].squeeze(0), enc["labels"][0]


def detr_collate(batch):
    pvs    = [b[0] for b in batch]
    labels = [b[1] for b in batch]
    h = max(p.shape[1] for p in pvs)
    w = max(p.shape[2] for p in pvs)
    pv = torch.zeros(len(pvs), 3, h, w)
    pm = torch.zeros(len(pvs), h, w, dtype=torch.long)
    for i, p in enumerate(pvs):
        _, ph, pw = p.shape
        pv[i, :, :ph, :pw] = p
        pm[i, :ph, :pw]    = 1
    return pv, pm, labels


detr_train = DataLoader(DetrBallDataset(ds4_train, DETR_PROC), batch_size=4,
                         shuffle=True,  collate_fn=detr_collate, num_workers=0)
detr_val   = DataLoader(DetrBallDataset(ds4_val,   DETR_PROC), batch_size=4,
                         shuffle=False, collate_fn=detr_collate, num_workers=0)
detr_test  = DataLoader(DetrBallDataset(ds4_test,  DETR_PROC), batch_size=1,
                         shuffle=False, collate_fn=detr_collate, num_workers=0)
print(f"DETR loaders: train={len(detr_train)} val={len(detr_val)} test={len(detr_test)}")

DETR loaders: train=49 val=7 test=27


In [39]:
!pip install timm

   ---------------------------------------- 0.0/2.6 MB ? eta -:--:--
   ---------------------------------------- 2.6/2.6 MB 14.9 MB/s  0:00:00


In [29]:
DETR_EPOCHS     = 50
DETR_PATIENCE   = 10
DETR_EVAL_EVERY = 5       # run val mAP every N epochs
DETR_MODEL_PATH = os.path.join(MODELS_DIR, "detr_final.pt")

detr_model = DetrForObjectDetection.from_pretrained(
    "facebook/detr-resnet-50", num_labels=1, ignore_mismatched_sizes=True
).to(DEVICE)
detr_opt = torch.optim.AdamW(detr_model.parameters(), lr=1e-4, weight_decay=1e-4)
detr_sch = torch.optim.lr_scheduler.CosineAnnealingLR(detr_opt, T_max=DETR_EPOCHS)

best_map50, pat_c = 0.0, 0
tr_losses, vl_map50s = [], []

for epoch in range(1, DETR_EPOCHS + 1):
    # -- training
    detr_model.train()
    ep_loss = 0.0
    for pv, pm, tgts in tqdm(detr_train, desc=f"DETR {epoch}/{DETR_EPOCHS}"):
        pv = pv.to(DEVICE)
        if pm is not None: pm = pm.to(DEVICE)
        tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
        loss = detr_model(pixel_values=pv, pixel_mask=pm, labels=tgts).loss
        detr_opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(detr_model.parameters(), 0.1)
        detr_opt.step()
        ep_loss += loss.item()
    detr_sch.step()
    avg_tr = ep_loss / max(len(detr_train), 1)
    tr_losses.append(avg_tr)

    # -- val mAP every DETR_EVAL_EVERY epochs
    if epoch % DETR_EVAL_EVERY == 0 or epoch == DETR_EPOCHS:
        detr_model.eval()
        val_preds, val_gts = [], []
        with torch.no_grad():
            for pv, pm, tgts in detr_val:
                pv = pv.to(DEVICE)
                if pm is not None: pm = pm.to(DEVICE)
                out = detr_model(pixel_values=pv, pixel_mask=pm)
                szs = torch.tensor([[pv.shape[-2], pv.shape[-1]]] * pv.shape[0])
                res = DETR_PROC.post_process_object_detection(
                    out, threshold=0.5, target_sizes=szs.to(DEVICE))
                for r, tgt in zip(res, tgts):
                    val_preds.append({
                        "boxes":  r["boxes"].cpu().numpy(),
                        "scores": r["scores"].cpu().numpy(),
                        "labels": r["labels"].cpu().numpy(),
                    })
                    gt = tgt["boxes"].cpu().numpy()
                    H, W = pv.shape[-2], pv.shape[-1]
                    if len(gt):
                        cx, cy, bw, bh = gt[:,0], gt[:,1], gt[:,2], gt[:,3]
                        gt_px = np.stack([
                            (cx - bw/2)*W, (cy - bh/2)*H,
                            (cx + bw/2)*W, (cy + bh/2)*H,
                        ], axis=1)
                    else:
                        gt_px = np.zeros((0, 4))
                    val_gts.append({"boxes": gt_px})

        val_metrics = evaluate_detections(val_preds, val_gts)
        map50 = val_metrics["mAP50"]
        vl_map50s.append((epoch, map50))
        print(f"Epoch {epoch:3d} | train_loss={avg_tr:.4f}  val_mAP50={map50:.4f}")

        if map50 > best_map50:
            best_map50, pat_c = map50, 0
            torch.save(detr_model.state_dict(), DETR_MODEL_PATH)
            print("  -> Best saved")
        else:
            pat_c += 1
            if pat_c >= DETR_PATIENCE:
                print("Early stopping.")
                break
    else:
        print(f"Epoch {epoch:3d} | train_loss={avg_tr:.4f}")

# -- loss / mAP curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tr_losses, label="Train loss")
epochs_eval, maps = zip(*vl_map50s) if vl_map50s else ([], [])
ax.plot(epochs_eval, maps, label="Val mAP@50", marker="o")
ax.set_title("DETR — Train Loss & Val mAP@50")
ax.legend(); plt.tight_layout()
plt.savefig("detr_loss_curve.png", dpi=120)
print("Saved -> detr_loss_curve.png")

[transformers] You passed `num_labels=1` which is incompatible to the `id2label` map of length `91`.
Loading weights: 100%|██████████| 530/530 [00:00<00:00, 14671.21it/s]
[transformers] DetrForObjectDetection LOAD REPORT from: facebook/detr-resnet-50
Key                                                            | Status     |                                                                                        
---------------------------------------------------------------+------------+----------------------------------------------------------------------------------------
model.backbone.model.layer3.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer1.0.downsample.1.num_batches_tracked | UNEXPECTED |                                                                                        
model.backbone.model.layer2.0.downsample.1.num_batches_tracked | UNEXPECTED |        

Epoch   1 | train_loss=1.9334


DETR 2/50: 100%|██████████| 49/49 [00:31<00:00,  1.57it/s]


Epoch   2 | train_loss=1.7089


DETR 3/50: 100%|██████████| 49/49 [00:31<00:00,  1.56it/s]


...

Epoch  48 | train_loss=0.5377


DETR 49/50: 100%|██████████| 49/49 [00:31<00:00,  1.56it/s]


Epoch  49 | train_loss=0.5354


DETR 50/50: 100%|██████████| 49/49 [00:31<00:00,  1.56it/s]


Epoch  50 | train_loss=0.5571  val_mAP50=0.9043
Saved -> detr_loss_curve.png


---
### 3.5 Faster R-CNN

`fasterrcnn_resnet50_fpn` (COCO pretrained). Classification head replaced for 2 classes.

In [30]:
from torchvision.models.detection import (fasterrcnn_resnet50_fpn,
                                           FasterRCNN_ResNet50_FPN_Weights)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

NUM_CLASSES      = 2
FRCNN_MODEL_PATH = os.path.join(MODELS_DIR, "frcnn_final.pt")

frcnn = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT)
frcnn.roi_heads.box_predictor = FastRCNNPredictor(
    frcnn.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
frcnn = frcnn.to(DEVICE)

frcnn_opt = torch.optim.AdamW(frcnn.parameters(), lr=1e-4, weight_decay=1e-4)
frcnn_sch = torch.optim.lr_scheduler.CosineAnnealingLR(frcnn_opt, T_max=30)



def det_epoch(model, loader, opt=None, train=True):
    model.train() if train else model.eval()
    total = 0.
    ctx   = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for imgs, tgts in tqdm(loader, desc="TR" if train else "VL"):
            imgs = [i.to(DEVICE) for i in imgs]
            tgts = [{k: v.to(DEVICE) for k, v in t.items()} for t in tgts]
            loss = sum(model(imgs, tgts).values())
            if train: opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item()
    return total / max(len(loader), 1)




In [31]:
print("Training Faster R-CNN...")
best_map50, fr_pat = 0.0, 0
fr_tr_loss, fr_vl_map50 = [], []
FRCNN_EVAL_EVERY = 5

for epoch in range(1, 31):
    tr = det_epoch(frcnn, train_loader, frcnn_opt, True)
    frcnn_sch.step()
    fr_tr_loss.append(tr)

    if epoch % FRCNN_EVAL_EVERY == 0 or epoch == 30:
        p_val, g_val = run_inference(frcnn, val_loader)
        val_metrics  = evaluate_detections(p_val, g_val)
        map50        = val_metrics["mAP50"]
        fr_vl_map50.append((epoch, map50))
        print(f"Epoch {epoch:3d} | tr={tr:.4f}  val_mAP50={map50:.4f}")
        if map50 > best_map50:
            best_map50, fr_pat = map50, 0
            torch.save(frcnn.state_dict(), FRCNN_MODEL_PATH)
            print("  -> Best")
        else:
            fr_pat += 1
            if fr_pat >= 3: print("Early stop."); break  # 3 * 5 = 15 epochs patience
    else:
        print(f"Epoch {epoch:3d} | tr={tr:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(fr_tr_loss, label="Train loss")
if fr_vl_map50:
    ep_fr, m_fr = zip(*fr_vl_map50)
    ax.plot(ep_fr, m_fr, label="Val mAP@50", marker="o")
ax.set_title("Faster R-CNN — Train Loss & Val mAP@50")
ax.legend(); plt.tight_layout()
plt.savefig("frcnn_loss_curve.png", dpi=120)
print("Saved -> frcnn_loss_curve.png")

Training Faster R-CNN...


TR: 100%|██████████| 49/49 [00:36<00:00,  1.36it/s]


Epoch   1 | tr=0.4475


TR: 100%|██████████| 49/49 [00:36<00:00,  1.34it/s]


Epoch   2 | tr=0.2480


...

Epoch  28 | tr=0.0390


TR: 100%|██████████| 49/49 [00:36<00:00,  1.34it/s]


Epoch  29 | tr=0.0377


Inference: 100%|██████████| 7/7 [00:02<00:00,  2.58it/s]


Epoch  30 | tr=0.0369  val_mAP50=0.9067
Saved -> frcnn_loss_curve.png


---
### 3.6 SSD

`ssd300_vgg16` (COCO pretrained). Classification head replaced for 2 classes.

In [32]:
from torchvision.models.detection import ssd300_vgg16, SSD300_VGG16_Weights
from torchvision.models.detection.ssd import SSDClassificationHead

SSD_MODEL_PATH = os.path.join(MODELS_DIR, "ssd_final.pt")

ssd_model = ssd300_vgg16(weights=SSD300_VGG16_Weights.DEFAULT)
ssd_model.head.classification_head = SSDClassificationHead(
    in_channels=[512, 1024, 512, 256, 256, 256],
    num_anchors=ssd_model.anchor_generator.num_anchors_per_location(),
    num_classes=NUM_CLASSES,
)
ssd_model = ssd_model.to(DEVICE)

ssd_opt = torch.optim.AdamW(ssd_model.parameters(), lr=1e-4, weight_decay=1e-4)
ssd_sch = torch.optim.lr_scheduler.CosineAnnealingLR(ssd_opt, T_max=30)


In [33]:
print("Training SSD300...")
best_map50, sd_pat = 0.0, 0
sd_tr_loss, sd_vl_map50 = [], []
SSD_EVAL_EVERY = 5

for epoch in range(1, 31):
    tr = det_epoch(ssd_model, train_loader, ssd_opt, True)
    ssd_sch.step()
    sd_tr_loss.append(tr)

    if epoch % SSD_EVAL_EVERY == 0 or epoch == 30:
        p_val, g_val = run_inference(ssd_model, val_loader)
        val_metrics  = evaluate_detections(p_val, g_val)
        map50        = val_metrics["mAP50"]
        sd_vl_map50.append((epoch, map50))
        print(f"Epoch {epoch:3d} | tr={tr:.4f}  val_mAP50={map50:.4f}")
        if map50 > best_map50:
            best_map50, sd_pat = map50, 0
            torch.save(ssd_model.state_dict(), SSD_MODEL_PATH)
            print("  -> Best")
        else:
            sd_pat += 1
            if sd_pat >= 3: print("Early stop."); break
    else:
        print(f"Epoch {epoch:3d} | tr={tr:.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sd_tr_loss, label="Train loss")
if sd_vl_map50:
    ep_sd, m_sd = zip(*sd_vl_map50)
    ax.plot(ep_sd, m_sd, label="Val mAP@50", marker="o")
ax.set_title("SSD300 — Train Loss & Val mAP@50")
ax.legend(); plt.tight_layout()
plt.savefig("ssd_loss_curve.png", dpi=120)
print("Saved -> ssd_loss_curve.png")

Training SSD300...


TR: 100%|██████████| 49/49 [00:11<00:00,  4.12it/s]


Epoch   1 | tr=3.9960


TR: 100%|██████████| 49/49 [00:11<00:00,  4.14it/s]


Epoch   2 | tr=2.8358


...

Epoch  18 | tr=1.6086


TR: 100%|██████████| 49/49 [00:11<00:00,  4.15it/s]


Epoch  19 | tr=1.4790


Inference: 100%|██████████| 7/7 [00:01<00:00,  5.34it/s]

Epoch  20 | tr=1.3545  val_mAP50=0.6859
Early stop.
Saved -> ssd_loss_curve.png


### 3.7 Additional YOLO Variants (YOLOv8m, YOLOv8l, YOLOv9s, YOLOv9m)

Four additional models trained with the same two-phase pipeline as YOLOv8s (pretrain on datasets 1+3, fine-tune on dataset 4). YOLOv8m and YOLOv8l are larger variants of YOLOv8s, while YOLOv9s and YOLOv9m are the successor architecture. All models use `single_cls=True` and the same hyperparameters.

In [34]:
# == Limpeza entre modelos: liberta RAM/VRAM e mata workers orfaos ==
import os, gc, torch
for _n in ['pretrain_model','finetune_model','pretrain_model_m',
          'finetune_model_m','rtdetr_pretrain','rtdetr_finetune']:
    globals().pop(_n, None)
gc.collect()
torch.cuda.empty_cache()
try:
    import psutil
    _kids = [c for c in psutil.Process(os.getpid()).children(recursive=True)
             if 'python' in (c.name() or '').lower()]
    for _c in _kids:
        try: _c.kill()
        except Exception: pass
    psutil.wait_procs(_kids, timeout=3)
    print(f"Limpeza OK -- {len(_kids)} workers orfaos terminados.")
except Exception as _e:
    print("Limpeza parcial (psutil indisponivel):", _e)

Limpeza OK -- 0 workers orfaos terminados.


In [ ]:
import os, shutil
from pathlib import Path
from ultralytics import YOLO
import matplotlib.pyplot as plt

# -- Paths
MODELS_DIR    = "models"
os.makedirs(MODELS_DIR, exist_ok=True)

PRETRAIN_YAML = "yolo_data/pretrain.yaml"
FINETUNE_YAML = "yolo_data/finetune.yaml"

# -- Hyperparameters
PRETRAIN_EPOCHS = 50
FINETUNE_EPOCHS = 100
IMG_SIZE        = 640
BATCH_SIZE      = 16   # reduce to 8 if VRAM is limited
PATIENCE        = 20
SEED            = 42
DEVICE          = 0    # GPU 0

def find_best(run_name):
    p = Path(f"runs/detect/{run_name}/weights/best.pt")
    if p.exists():
        return str(p)
    raise FileNotFoundError(f"best.pt not found for run: {run_name}")


def train_two_phase(model_name, run_prefix):
    """Generic two-phase train: pretrain on ds1+3, finetune on ds4."""
    print("=" * 60)
    print(f"{model_name} — Phase 1: pretrain on datasets 1, 3")
    print("=" * 60)
    model = YOLO(f"{model_name}.pt")
    model.train(
        data=PRETRAIN_YAML, epochs=PRETRAIN_EPOCHS, imgsz=IMG_SIZE,
        batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
        name=f"{run_prefix}_pretrain", exist_ok=True, single_cls=True,
        device=DEVICE, workers=4, cache=False,
    )
    pretrain_best = find_best(f"{run_prefix}_pretrain")
    shutil.copy(pretrain_best, os.path.join(MODELS_DIR, f"{run_prefix}_pretrained.pt"))

    print("=" * 60)
    print(f"{model_name} — Phase 2: fine-tune on dataset 4")
    print("=" * 60)
    model = YOLO(pretrain_best)
    model.train(
        data=FINETUNE_YAML, epochs=FINETUNE_EPOCHS, imgsz=IMG_SIZE,
        batch=BATCH_SIZE, patience=PATIENCE, seed=SEED,
        name=f"{run_prefix}_finetune", exist_ok=True, single_cls=True,
        device=DEVICE, workers=4, cache=False,
        lr0=1e-4, lrf=0.01,
    )
    final_best = find_best(f"{run_prefix}_finetune")
    shutil.copy(final_best, os.path.join(MODELS_DIR, f"{run_prefix}_final.pt"))

    # -- Evaluation
    eval_model = YOLO(final_best)
    metrics = eval_model.val(
        data=FINETUNE_YAML, split="test",
        imgsz=IMG_SIZE, batch=BATCH_SIZE, device=DEVICE,
    )
    results = {
        "mAP50":     float(metrics.box.map50),
        "mAP50-95":  float(metrics.box.map),
        "Precision": float(metrics.box.mp),
        "Recall":    float(metrics.box.mr),
    }
    print(f"{model_name} results: {results}")
    return results, final_best


# ── YOLOv8m ──────────────────────────────────────────────────────────────────
yolov8m_results, yolov8m_best = train_two_phase("yolov8m", "yolov8m")

# ── YOLOv8l ──────────────────────────────────────────────────────────────────
yolov8l_results, yolov8l_best = train_two_phase("yolov8l", "yolov8l")

# ── YOLOv9s ──────────────────────────────────────────────────────────────────
yolov9s_results, yolov9s_best = train_two_phase("yolov9s", "yolov9s")

# ── YOLOv9m ──────────────────────────────────────────────────────────────────
yolov9m_results, yolov9m_best = train_two_phase("yolov9m", "yolov9m")

# ── Sample predictions for each model ────────────────────────────────────────
test_imgs = sorted(Path("yolo_data").glob("**/images/test/*.jpg"))[:6]

for model_name, best_path, label in [
    ("yolov8m", yolov8m_best, "YOLOv8m"),
    ("yolov8l", yolov8l_best, "YOLOv8l"),
    ("yolov9s", yolov9s_best, "YOLOv9s"),
    ("yolov9m", yolov9m_best, "YOLOv9m"),
]:
    model = YOLO(best_path)
    preds = model.predict(
        source=[str(p) for p in test_imgs],
        imgsz=IMG_SIZE, conf=0.25, verbose=False, device=DEVICE,
    )
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    for ax, r in zip(axes.flatten(), preds):
        ax.imshow(r.plot()[:, :, ::-1])
        ax.set_title(f"{Path(r.path).name}\n{len(r.boxes)} balls", fontsize=8)
        ax.axis("off")
    for ax in axes.flatten()[len(preds):]:
        ax.axis("off")
    plt.suptitle(f"{label} — Sample Test Predictions")
    plt.tight_layout()
    out_file = f"{model_name}_predictions.png"
    plt.savefig(out_file, dpi=150)
    print(f"Saved -> {out_file}")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n" + "=" * 65)
print("New models — Results Summary")
print("=" * 65)
print(f"{'Model':<12} | {'mAP@50':>8} | {'mAP@50-95':>10} | {'Precision':>10} | {'Recall':>8}")
print("-" * 65)
for name, res in [
    ("YOLOv8m",  yolov8m_results),
    ("YOLOv8l",  yolov8l_results),
    ("YOLOv9s",  yolov9s_results),
    ("YOLOv9m",  yolov9m_results),
]:
    print(f"{name:<12} | {res['mAP50']:>8.4f} | {res['mAP50-95']:>10.4f} | "
          f"{res['Precision']:>10.4f} | {res['Recall']:>8.4f}")
print("=" * 65)


# ── Save checkpoints ──────────────────────────────────────────────────────────
save_checkpoint("yolov8m_results", yolov8m_results)
save_checkpoint("yolov8l_results", yolov8l_results)
save_checkpoint("yolov9s_results", yolov9s_results)
save_checkpoint("yolov9m_results", yolov9m_results)

yolov8m — Phase 1: pretrain on datasets 1, 3
Ultralytics 8.4.60  Python-3.13.11 torch-2.12.0+cu132 CUDA:0 (NVIDIA GeForce RTX 5060 Ti, 16311MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_data/pretrain.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8m_pretrain, nbs=64, nms=False, opset=None, o

### 3.8 Quantitative Comparison

All ten models evaluated on the **dataset-4 test split**. Results include the original six models (YOLOv8s with and without masking, RT-DETR, DETR, Faster R-CNN, SSD300) plus four additional YOLO variants (YOLOv8m, YOLOv8l, YOLOv9s, YOLOv9m) trained with the same two-phase pipeline (pretrain on datasets 1+3, fine-tune on dataset 4).

In [ ]:
# ── Carregar checkpoints se variáveis não estiverem em memória ──────────────
_ckpt = load_checkpoints()
_needed = [
    "yolo_no_mask_results", "yolo_mask_results", "rtdetr_results",
    "detr_results", "frcnn_results", "ssd_results",
    "yolov8m_results", "yolov8l_results", "yolov9s_results", "yolov9m_results",
]
_ns = vars()
for _key in _needed:
    if _key not in _ns or _ns[_key] is None:
        if _key in _ckpt:
            exec(f"{_key} = _ckpt['{_key}']")
            print(f"  ↺ loaded {_key} from checkpoint")
        else:
            raise RuntimeError(
                f"Missing: {_key}. Run the corresponding training/eval cell first."
            )

RESULTS_FILE = "task3_results.txt"
all_results = {
    "YOLOv8s (no mask)": yolo_no_mask_results,
    "YOLOv8s (masked)":  yolo_mask_results,
    "RT-DETR":           rtdetr_results,
    "DETR":              detr_results,
    "Faster R-CNN":      frcnn_results,
    "SSD300":            ssd_results,
    "YOLOv8m":           yolov8m_results,
    "YOLOv8l":           yolov8l_results,
    "YOLOv9s":           yolov9s_results,
    "YOLOv9m":           yolov9m_results,
}
keys = ["mAP50", "mAP50-95", "Precision", "Recall"]

# -- Print table
sep = "=" * 75
print(f"\n{sep}")
print("TASK 3 — Ball Detection Results (all models, ds4 test split)")
print(sep)
print(f"{'Model':<22} | {'mAP@50':>8} | {'mAP@50-95':>10} | {'Precision':>10} | {'Recall':>8}")
print("-" * 75)
for name, r in all_results.items():
    print(f"{name:<22} | {r['mAP50']:>8.4f} | {r['mAP50-95']:>10.4f} | "
          f"{r['Precision']:>10.4f} | {r['Recall']:>8.4f}")
print(sep)

with open(RESULTS_FILE, "w") as f:
    f.write("\n".join([sep,
        "TASK 3 — Ball Detection Results", sep,
        f"{'Model':<22} | {'mAP@50':>8} | {'mAP@50-95':>10} | {'Precision':>10} | {'Recall':>8}",
        "-" * 75
    ] + [
        f"{n:<22} | {r['mAP50']:>8.4f} | {r['mAP50-95']:>10.4f} | "
        f"{r['Precision']:>10.4f} | {r['Recall']:>8.4f}"
        for n, r in all_results.items()
    ] + [sep]))
print(f"Saved -> {RESULTS_FILE}")

# -- Bar chart (10 models, adjusted bar width and colors)
n       = len(all_results)
x       = np.arange(len(keys))
bw      = 0.08
offsets = np.linspace(-(n-1)/2, (n-1)/2, n) * bw
colors  = [
    "#4C72B0", "#DD8452", "#55A868", "#C44E52",
    "#8172B2", "#937860", "#DA8BC3", "#8C8C8C",
    "#CCB974", "#64B5CD",
]

fig, ax = plt.subplots(figsize=(18, 6))
for i, (name, res) in enumerate(all_results.items()):
    vals = [res[k] for k in keys]
    bars = ax.bar(x + offsets[i], vals, bw, label=name, color=colors[i])
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.2f}", ha="center", va="bottom", fontsize=4, rotation=90)

ax.set_xticks(x)
ax.set_xticklabels(["mAP@50", "mAP@50-95", "Precision", "Recall"], fontsize=11)
ax.set_ylim(0, 1.2)
ax.set_ylabel("Score")
ax.set_title("Ball Detection — Quantitative Comparison", fontsize=13)
ax.legend(loc="upper right", fontsize=7, ncol=2)
plt.tight_layout()
plt.savefig("detection_comparison.png", dpi=150)
print("Saved -> detection_comparison.png")

---
## 4 — Table Retrieval

For each **query image** from the dataset-4 test split we:
1. Detect ball centres with YOLOv8 (`yolov8s_final.pt`).
2. Use the **table corners** returned by `segment_table` to compute a homography
   that maps pixel coordinates → canonical 1×1 court space.
3. Compare the normalised centre distribution against every training image
   using **Earth Mover's Distance** (EMD ≈ sum of 1-D Wasserstein distances).
4. Return the **top-3** most similar images.

### 4.1 Ball Position Extraction

In [ ]:
RETRIEVAL_WEIGHTS = os.path.join(MODELS_DIR, "yolov8s_final.pt")
DETECT_CONF       = 0.25

retrieval_model = YOLO(RETRIEVAL_WEIGHTS)


def get_ball_centers(image_path, model, conf=DETECT_CONF, imgsz=640):
    """Run detector; return list of (cx_px, cy_px) tuples."""
    results = model.predict(source=str(image_path), conf=conf,
                             imgsz=imgsz, verbose=False)
    centers = []
    for r in results:
        for box in r.boxes.xyxy.cpu().numpy():
            x1, y1, x2, y2 = box
            centers.append(((x1+x2)/2, (y1+y2)/2))
    return centers


if ds4_test:
    sample_path = ds4_test[0]["image_path"]
    print(f"Test image : {sample_path}")
    print(f"Centres    : {get_ball_centers(sample_path, retrieval_model)[:5]}")

### 4.2 Normalized Court Mapping via Homography

The 4 table corners returned by `segment_table` are used to build a homography $H$:

$$\text{pixel corners} \xrightarrow{H} (0,0),\ (1,0),\ (1,1),\ (0,1)$$

Ball centres are then expressed in $[0,1]^2$, making them camera-angle invariant.

In [ ]:
CANONICAL = np.array([[0.,0.],[1.,0.],[1.,1.],[0.,1.]], dtype=np.float32)


def get_table_corners(image_path, entry):
    """
    Detect table corners via segment_table.
    Falls back to image bounding box if the image cannot be read.
    """
    img = cv2.imread(str(image_path))
    if img is not None:
        _, corners = segment_table(img)
        return corners
    W, H = float(entry["width"]), float(entry["height"])
    return np.array([[0,0],[W,0],[W,H],[0,H]], dtype=np.float32)


def compute_homography(corners_px):
    H, _ = cv2.findHomography(corners_px, CANONICAL)
    return H


def normalize_centers(centers_px, H_mat):
    if not centers_px or H_mat is None:
        return []
    pts  = np.array([[c[0], c[1]] for c in centers_px],
                    dtype=np.float32).reshape(-1, 1, 2)
    norm = cv2.perspectiveTransform(pts, H_mat).reshape(-1, 2)
    return list(map(tuple, np.clip(norm, 0., 1.)))


if ds4_test:
    e0      = ds4_test[0]
    corners = get_table_corners(e0["image_path"], e0)
    H_mat   = compute_homography(corners)
    ctrs    = get_ball_centers(e0["image_path"], retrieval_model)
    print(f"Pixel centres   : {ctrs[:3]}")
    print(f"Normalised (u,v): {normalize_centers(ctrs, H_mat)[:3]}")

### 4.3 Similarity Search — EMD

$$\text{EMD}(Q,G) = W_1(Q_x,G_x) + W_1(Q_y,G_y)$$

Gallery built from all dataset-4 **training** images.

In [ ]:
print(f"Building gallery from {len(ds4_train)} training images...")

gallery = []
for entry in tqdm(ds4_train, desc="Gallery"):
    if not os.path.isfile(entry["image_path"]): continue
    corners = get_table_corners(entry["image_path"], entry)
    H_mat   = compute_homography(corners)
    ctrs    = get_ball_centers(entry["image_path"], retrieval_model)
    gallery.append({"entry": entry,
                    "norm_centers": normalize_centers(ctrs, H_mat)})

print(f"Gallery ready: {len(gallery)} images.")


def emd_dist(qc, gc):
    if not qc or not gc: return 1.0
    q, g = np.array(qc), np.array(gc)
    return float(wasserstein_distance(q[:,0], g[:,0]) +
                 wasserstein_distance(q[:,1], g[:,1]))


def retrieve_top_k(query_entry, gallery, k=3):
    corners  = get_table_corners(query_entry["image_path"], query_entry)
    H_mat    = compute_homography(corners)
    ctrs     = get_ball_centers(query_entry["image_path"], retrieval_model)
    query_nc = normalize_centers(ctrs, H_mat)
    dists    = sorted([(emd_dist(query_nc, g["norm_centers"]), g) for g in gallery],
                      key=lambda x: x[0])
    return query_nc, dists[:k]


if ds4_test:
    _, top3 = retrieve_top_k(ds4_test[0], gallery)
    print(f"\nQuery: {ds4_test[0]['file_name']}")
    for dist, g in top3:
        print(f"  EMD={dist:.4f}  {g['entry']['file_name']}")

### 4.4 Qualitative Evaluation

**3 good** (lowest EMD) and **3 bad** (highest EMD) retrieval examples.  
Each row: **query** | **rank-1** | **rank-2** | **rank-3** — red dots = ball centres.

In [ ]:
print("Running retrieval on all test queries...")

test_entries  = [e for e in ds4_test if os.path.isfile(e["image_path"])]
query_results = []
for e in tqdm(test_entries, desc="Query"):
    q_nc, top3 = retrieve_top_k(e, gallery, k=3)
    query_results.append((top3[0][0] if top3 else 1., e, top3, q_nc))

query_results.sort(key=lambda x: x[0])
good_q = query_results[:3]
bad_q  = query_results[-3:]

all_emds = [r[0] for r in query_results]
print(f"Queries={len(query_results)}  "
      f"best={min(all_emds):.4f}  "
      f"median={np.median(all_emds):.4f}  "
      f"worst={max(all_emds):.4f}")


def draw_centers(img_bgr, centers_px):
    out = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB).copy()
    for cx, cy in centers_px:
        cv2.circle(out, (int(cx), int(cy)), 6, (255, 0, 0), -1)
    return out


def show_panel(ax_row, query_entry, top3, prefix):
    q_bgr  = cv2.imread(query_entry["image_path"])
    q_ctrs = get_ball_centers(query_entry["image_path"], retrieval_model)
    q_rgb  = draw_centers(q_bgr, q_ctrs) if q_bgr is not None \
             else np.zeros((100, 100, 3), dtype=np.uint8)
    ax_row[0].imshow(q_rgb)
    ax_row[0].set_title(f"{prefix}\n{query_entry['file_name'][:30]}", fontsize=7)
    ax_row[0].axis("off")
    for j, (dist, g) in enumerate(top3):
        g_bgr  = cv2.imread(g["entry"]["image_path"])
        g_ctrs = get_ball_centers(g["entry"]["image_path"], retrieval_model)
        g_rgb  = draw_centers(g_bgr, g_ctrs) if g_bgr is not None \
                 else np.zeros((100, 100, 3), dtype=np.uint8)
        ax_row[j+1].imshow(g_rgb)
        ax_row[j+1].set_title(f"Rank {j+1}  EMD={dist:.3f}\n"
                               f"{g['entry']['file_name'][:30]}", fontsize=7)
        ax_row[j+1].axis("off")


# Good examples
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, (emd, qe, top3, _) in enumerate(good_q):
    show_panel(axes[i], qe, top3, f"GOOD  EMD={emd:.3f}")
plt.suptitle("Table Retrieval — Good Examples (lowest EMD)", fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig("retrieval_good.png", dpi=130, bbox_inches="tight")
print("Saved -> retrieval_good.png")

# Bad examples
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
for i, (emd, qe, top3, _) in enumerate(bad_q):
    show_panel(axes[i], qe, top3, f"BAD   EMD={emd:.3f}")
plt.suptitle("Table Retrieval — Bad Examples (highest EMD)", fontsize=13, y=1.01)
plt.tight_layout(); plt.savefig("retrieval_bad.png", dpi=130, bbox_inches="tight")
print("Saved -> retrieval_bad.png")

# EMD histogram
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(all_emds, bins=20, color="#4C72B0", edgecolor="white")
ax.axvline(np.median(all_emds), color="red",  ls="--",
           label=f"Median={np.median(all_emds):.3f}")
ax.axvline(np.mean(all_emds),   color="lime", ls="--",
           label=f"Mean  ={np.mean(all_emds):.3f}")
ax.set_xlabel("Best-match EMD"); ax.set_ylabel("Count")
ax.set_title("Retrieval — EMD Distribution"); ax.legend()
plt.tight_layout(); plt.savefig("retrieval_emd_distribution.png", dpi=120)
print("Saved -> retrieval_emd_distribution.png")

print("\n" + "=" * 50)
print("Task 3 — Retrieval Summary")
print("=" * 50)
print(f"  Gallery  : {len(gallery)}  |  Queries: {len(query_results)}")
print(f"  Mean EMD : {np.mean(all_emds):.4f}")
print(f"  Median   : {np.median(all_emds):.4f}")
print(f"  Min/Max  : {min(all_emds):.4f} / {max(all_emds):.4f}")
print("=" * 50)